# Fase 1: Extracción de datos de películas

In [4]:
import requests
import pandas as pd
import numpy as np

In [5]:
url = "https://beta.adalab.es/resources/apis/pelis/pelis.json"
datos = requests.get(url)
datos.status_code


200

In [6]:
datos_pelis = datos.json()
len(datos_pelis)
datos_pelis [0]

{'id': 1,
 'titulo': 'The Godfather',
 'año': 1972,
 'duracion': 175,
 'genero': 'Crimen',
 'adultos': False,
 'subtitulos': ['es', 'en']}

In [18]:
lista_peliculas = []

for pelicula in datos_pelis:
    nueva_pelicula = {
        'titulo':pelicula['titulo'],
        'año': pelicula ['año'],
        'duracion': pelicula ['duracion'],
        'genero': pelicula ['genero'],
        'adultos': pelicula ['adultos'],
        'subtitulos': pelicula ['subtitulos']
    }
    lista_peliculas.append(nueva_pelicula)

In [19]:
tabla_pelis = pd.DataFrame(lista_peliculas)
tabla_pelis

,titulo,año,duracion,genero,adultos,subtitulos
0,The Godfather,1972,175,Crimen,False,"[es, en]"
1,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
2,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
3,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
4,The Dark Knight,2008,152,Acción,False,"[es, en]"
...,...,...,...,...,...,...
95,La vita è bella,1997,116,Drama,False,"[es, en, it]"
96,Requiem for a Dream,2000,102,Drama,True,"[es, en]"
97,Memento,2000,113,Thriller,True,"[es, en]"
98,Eternal Sunshine of the Spotless Mind,2004,108,Drama,False,"[es, en]"


In [20]:
tabla_pelis.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   titulo      100 non-null    str   
 1   año         100 non-null    int64 
 2   duracion    100 non-null    int64 
 3   genero      100 non-null    str   
 4   adultos     100 non-null    bool  
 5   subtitulos  100 non-null    object
dtypes: bool(1), int64(2), object(1), str(2)
memory usage: 4.1+ KB


# Fase 3: inserción de los datos en la base de datos

* La fase 2 se ha realizado en el archivo SQL "ejercicio-1-api"

In [ ]:
pip install python-dotenv

### Conexión con MySQL

In [10]:
import os
from dotenv import load_dotenv

load_dotenv('/Users/beatriz/Documents/ADALAB DATA ANALYTICS/1. Ejercicios/Modulos/modulo_2/.env')
password_sql = os.getenv("PASS_SQL")

In [11]:
print(password_sql is not None)

True


In [ ]:
!pip install mysql-connector-python

In [12]:
import mysql.connector
from mysql.connector import Error

try:
    connection = mysql.connector.connect(
        host = '127.0.0.1',
        user = 'root',
        password = password_sql,
    )
    print('Conexion exitosa')
except Error as e:
    print ({e})

Conexion exitosa


In [14]:
try:
    cursor = connection.cursor()
    query ='USE peliculas_evaluacion'
    cursor.execute(query)
    print ("Query creada con éxito")

except Error as e:
    print(e)

Query creada con éxito


#### Limpieza de columna subtítulos

In [22]:
subtitulos_limpios = []

for subtitulo in tabla_pelis["subtitulos"]:
    subtitulo_texto = ", ".join(subtitulo)
    subtitulos_limpios.append(subtitulo_texto)

tabla_pelis["subtitulos"] = subtitulos_limpios

In [23]:
tabla_pelis.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   titulo      100 non-null    str  
 1   año         100 non-null    int64
 2   duracion    100 non-null    int64
 3   genero      100 non-null    str  
 4   adultos     100 non-null    bool 
 5   subtitulos  100 non-null    str  
dtypes: bool(1), int64(2), str(3)
memory usage: 4.1 KB


### Inserción de datos en la base de datos

In [24]:
cursor.execute ('USE peliculas_evaluacion')

query_insert = '''INSERT INTO peliculas (titulo, anio, duracion, genero, adultos, subtitulos) 
VALUES (%s, %s, %s, %s, %s, %s)'''

datos_tabla = tabla_pelis.values.tolist()

cursor.executemany(query_insert, datos_tabla)
connection.commit()

print(f"Se han insertado nuevos {cursor.rowcount} valores")

Se han insertado nuevos 100 valores


# Consultas

Cuántas películas tienen una duración superior a 120 minutos?

In [39]:
query_consulta1 = '''SELECT titulo, anio, duracion, genero, adultos, subtitulos
FROM peliculas
WHERE duracion >120
ORDER BY duracion;'''

cursor.execute(query_consulta1)
resultados1 = cursor.fetchall()
columnas = cursor.column_names

df_resultados1 = pd.DataFrame(resultados1,columns=columnas)
df_resultados1


,titulo,anio,duracion,genero,adultos,subtitulos
0,Star Wars: A New Hope,1977,121,Ciencia ficción,0,"es, en"
1,The Exorcist,1973,122,Terror,1,"es, en"
2,American Beauty,1999,122,Drama,1,"es, en"
3,Joker,2019,122,Drama,1,"es, en"
4,No Country for Old Men,2007,122,Crimen,1,"es, en"
5,Amélie,2001,122,Romance,0,"es, en, fr"
6,The Shape of Water,2017,123,Fantasía,1,"es, en"
7,Star Wars: The Empire Strikes Back,1980,124,Ciencia ficción,0,"es, en"
8,Spirited Away,2001,125,Animación,0,"es, en, jp"
9,Her,2013,126,Drama,0,"es, en"


Cuántas películas incluyen subtítulos en español?

In [40]:
query_consulta2 = '''SELECT titulo, anio, duracion, genero, adultos, subtitulos
FROM peliculas
WHERE subtitulos LIKE '%es%';'''

cursor.execute(query_consulta2)
resultados2 = cursor.fetchall()
columnas2 = cursor.column_names

df_resultados2 = pd.DataFrame(resultados2,columns=columnas2)
df_resultados2

,titulo,anio,duracion,genero,adultos,subtitulos
0,The Godfather,1972,175,Crimen,0,"es, en"
1,The Godfather Part II,1974,202,Crimen,0,"es, en"
2,Pulp Fiction,1994,154,Crimen,1,"es, en"
3,Forrest Gump,1994,142,Drama,0,"es, en, fr"
4,The Dark Knight,2008,152,Acción,0,"es, en"
...,...,...,...,...,...,...
95,La vita è bella,1997,116,Drama,0,"es, en, it"
96,Requiem for a Dream,2000,102,Drama,1,"es, en"
97,Memento,2000,113,Thriller,1,"es, en"
98,Eternal Sunshine of the Spotless Mind,2004,108,Drama,0,"es, en"
